# Bootstrap Colab — Gen 2 (BERT)

Notebook de orquestração para fine-tuning dos encoders BERT em Colab
(T4 ou L4). Mantém o código reutilizável em `src/ptbr_market/gen2_bert.py`
— este notebook só faz setup + chama `scripts/run_gen2.py`.

**Pré-requisitos no Drive** (upload manual antes da primeira execução):

```
MyDrive/ptbr-market-classification/
  artifacts/
    splits/
      train.parquet
      val.parquet
      test.parquet
      metadata.json
```

Os runs vão escrever de volta em `MyDrive/ptbr-market-classification/artifacts/runs/`
automaticamente (via `PTBR_ARTIFACTS_ROOT` apontando para o Drive).

**Runtime recomendado:** GPU T4 (grátis) para `bertimbau-base`,
`finbert-ptbr`, `distilbertimbau`, `xlmr-base`, `deb3rta-base`.
`bertimbau-large` exige L4 ou precisa de `--batch-size 4 --grad-accum 8`
em T4.

## 1. Montar o Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clonar o repositório

Clone é sempre fresco (não persiste no Drive) — garante que estamos
rodando exatamente o commit da branch escolhida.

In [ ]:
REPO_URL = 'https://github.com/almeidadm/ptbr-market-classification.git'
BRANCH = 'main'

!rm -rf /content/ptbr-market-classification
!git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/ptbr-market-classification
%cd /content/ptbr-market-classification
!git log -1 --oneline

## 3. Instalar dependências

Instalação manual dos extras `gen2` — Colab já vem com `torch` CUDA, mas
versões podem divergir. `pip install -e .` instala o projeto em modo
editável (deixa os imports `from ptbr_market import ...` funcionando).

In [ ]:
!pip install -q -e .
!pip install -q 'transformers>=4.40' 'accelerate>=0.30' 'datasets>=2.20'

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 4. Apontar artefatos para o Drive

`PTBR_ARTIFACTS_ROOT` faz com que `new_run_dir` grave em `artifacts/runs/`
**dentro do Drive** — os runs sobrevivem ao encerramento do Colab.

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/ptbr-market-classification'
os.environ['PTBR_ARTIFACTS_ROOT'] = f'{DRIVE_ROOT}/artifacts'

!ls -la $PTBR_ARTIFACTS_ROOT/splits/

## 5. Rodar o modelo

Cada modelo roda **duas vezes**: uma binária (`--target-mode binary`,
default) e uma multiclasse com `top7_plus_other` (`--target-mode
multiclass --collapse-scheme top7_plus_other`). A comparação bin×mc8
é o mesmo protocolo aplicado a todas as gerações — replica a pergunta
"a decomposição da classe negativa ajuda?" respondida em Gen 1.

Uma célula por modelo — facilita executar isolado quando a sessão
Colab estiver apertada.

In [ ]:
!python scripts/run_gen2.py --model bertimbau-base
!python scripts/run_gen2.py --model bertimbau-base --target-mode multiclass --collapse-scheme top7_plus_other

In [ ]:
# !python scripts/run_gen2.py --model finbert-ptbr
# !python scripts/run_gen2.py --model finbert-ptbr --target-mode multiclass --collapse-scheme top7_plus_other

In [ ]:
# !python scripts/run_gen2.py --model distilbertimbau
# !python scripts/run_gen2.py --model distilbertimbau --target-mode multiclass --collapse-scheme top7_plus_other

In [ ]:
# !python scripts/run_gen2.py --model bertimbau-large --batch-size 4 --grad-accum 8
# !python scripts/run_gen2.py --model bertimbau-large --batch-size 4 --grad-accum 8 --target-mode multiclass --collapse-scheme top7_plus_other

In [ ]:
# !python scripts/run_gen2.py --model xlmr-base
# !python scripts/run_gen2.py --model xlmr-base --target-mode multiclass --collapse-scheme top7_plus_other

In [ ]:
# !python scripts/run_gen2.py --model deb3rta-base
# !python scripts/run_gen2.py --model deb3rta-base --target-mode multiclass --collapse-scheme top7_plus_other

## 6. Conferir o run que saiu

In [ ]:
!ls -lt $PTBR_ARTIFACTS_ROOT/runs/ | head

import json
from pathlib import Path

runs_dir = Path(os.environ['PTBR_ARTIFACTS_ROOT']) / 'runs'
latest = max((d for d in runs_dir.iterdir() if d.is_dir() and '__gen2__' in d.name),
             key=lambda d: d.stat().st_mtime)
print(f'\nLatest Gen 2 run: {latest.name}')
meta = json.loads((latest / 'metadata.json').read_text())
print(json.dumps(meta['metrics'], indent=2))
print(json.dumps(meta['efficiency'], indent=2))
print(f"threshold = {meta['threshold']['value']}")

## 7. Sincronizar localmente (depois, no host)

Como os runs ficam no Drive, basta baixá-los via Google Drive Desktop ou
`gdown`/`rclone` no host. Opção simples:

```bash
# No host local, depois que os runs terminarem no Colab:
rclone sync drive:ptbr-market-classification/artifacts/runs ./artifacts/runs \
    --include '*__gen2__*'
```

Ou simplesmente copiar os diretórios via a UI do Drive.